In [ ]:
!pip install ultralytics filterpy scikit-image lap opencv-python-headless


In [3]:

import numpy as np
from scipy.optimize import linear_sum_assignment

class KalmanFilter:
    def __init__(self, dim_x, dim_z):
        self.x = np.zeros((dim_x, 1))
        self.P = np.eye(dim_x) * 1000.
        self.F = np.eye(dim_x)
        self.H = np.zeros((dim_z, dim_x))
        self.R = np.eye(dim_z)
        self.Q = np.eye(dim_x)

    def predict(self):
        self.x = np.dot(self.F, self.x)
        self.P = np.dot(np.dot(self.F, self.P), self.F.T) + self.Q

    def update(self, z):
        y = z - np.dot(self.H, self.x)
        S = np.dot(np.dot(self.H, self.P), self.H.T) + self.R
        K = np.dot(np.dot(self.P, self.H.T), np.linalg.inv(S))
        self.x = self.x + np.dot(K, y)
        self.P = self.P - np.dot(np.dot(K, self.H), self.P)

class Track:
    def __init__(self, bbox, track_id):
        self.id = track_id
        self.hits = 0
        self.age = 0
        self.time_since_update = 0
        self.bbox = bbox
        self.kf = KalmanFilter(dim_x=7, dim_z=4)
        self.kf.F = np.array([[1,0,0,0,1,0,0],[0,1,0,0,0,1,0],[0,0,1,0,0,0,1],[0,0,0,1,0,0,0],[0,0,0,0,1,0,0],[0,0,0,0,0,1,0],[0,0,0,0,0,0,1]])
        self.kf.H = np.array([[1,0,0,0,0,0,0],[0,1,0,0,0,0,0],[0,0,1,0,0,0,0],[0,0,0,1,0,0,0]])
        (x, y, w, h) = bbox
        self.kf.x = np.array([[x+w/2],[y+h/2],[w*h],[w/h],[0],[0],[0]])
        self.kf.P *= 10.; self.kf.R *= 10.; self.kf.Q[4:, 4:] *= 0.01

    def update(self, bbox):
        self.time_since_update = 0
        self.hits += 1
        self.bbox = bbox
        (x, y, w, h) = bbox
        self.kf.update(np.array([[x+w/2],[y+h/2],[w*h],[w/h]]))

    def predict(self):
        self.kf.predict()
        self.age += 1
        self.time_since_update += 1
        return self.get_bbox()

    def get_bbox(self):
        x, y, s, r = self.kf.x[:4].flatten()
        w = np.sqrt(s * r); h = s / w
        return [x - w / 2, y - h / 2, w, h]

class Sort:
    def __init__(self, max_age=1, min_hits=3):
        self.max_age = max_age
        self.min_hits = min_hits
        self.tracks = []; self.frame_count = 0; self.track_id_counter = 0

    def update(self, detections):
        self.frame_count += 1
        predicted_tracks = [track.predict() for track in self.tracks]
        detections = np.array(detections) if len(detections) > 0 else np.empty((0, 4))
        predicted_tracks = np.array(predicted_tracks) if len(predicted_tracks) > 0 else np.empty((0, 4))
        iou_matrix = self._iou_batch(predicted_tracks, detections)
        row_ind, col_ind = linear_sum_assignment(-iou_matrix)
        for r, c in zip(row_ind, col_ind):
            if iou_matrix[r, c] >= 0.3: self.tracks[r].update(detections[c])
        unmatched_detections = [d for i, d in enumerate(detections) if i not in col_ind]
        for bbox in unmatched_detections:
            self.track_id_counter += 1
            self.tracks.append(Track(bbox, self.track_id_counter))
        self.tracks = [t for t in self.tracks if t.time_since_update <= self.max_age]
        output_tracks = []
        for track in self.tracks:
            if track.hits >= self.min_hits or self.frame_count <= self.min_hits:
                output_tracks.append(np.concatenate((track.get_bbox(), [track.id])).reshape(1, -1))
        return np.concatenate(output_tracks) if len(output_tracks) > 0 else np.empty((0, 5))

    def _iou_batch(self, bboxes1, bboxes2):
        if bboxes1.shape[0] == 0 or bboxes2.shape[0] == 0: return np.zeros((bboxes1.shape[0], bboxes2.shape[0]))
        bboxes1 = np.hstack((bboxes1[:, :2], bboxes1[:, :2] + bboxes1[:, 2:]))
        bboxes2 = np.hstack((bboxes2[:, :2], bboxes2[:, :2] + bboxes2[:, 2:]))
        x1 = np.maximum(bboxes1[:, None, 0], bboxes2[:, 0]); y1 = np.maximum(bboxes1[:, None, 1], bboxes2[:, 1])
        x2 = np.minimum(bboxes1[:, None, 2], bboxes2[:, 2]); y2 = np.minimum(bboxes1[:, None, 3], bboxes2[:, 3])
        intersection = np.maximum(0, x2 - x1) * np.maximum(0, y2 - y1)
        union = (bboxes1[:, 2] - bboxes1[:, 0]) * (bboxes1[:, 3] - bboxes1[:, 1])[:, None] + (bboxes2[:, 2] - bboxes2[:, 0]) * (bboxes2[:, 3] - bboxes2[:, 1]) - intersection
        return intersection / union


In [ ]:
import cv2
import numpy as np
from ultralytics import YOLO
from google.colab import files
from scipy.optimize import linear_sum_assignment

# KalmanFilter, Track, and Sort classes
class KalmanFilter:
    def __init__(self, dim_x, dim_z):
        self.x = np.zeros((dim_x, 1))
        self.P = np.eye(dim_x) * 1000.
        self.F = np.eye(dim_x)
        self.H = np.zeros((dim_z, dim_x))
        self.R = np.eye(dim_z)
        self.Q = np.eye(dim_x)

    def predict(self):
        self.x = np.dot(self.F, self.x)
        self.P = np.dot(np.dot(self.F, self.P), self.F.T) + self.Q

    def update(self, z):
        y = z - np.dot(self.H, self.x)
        S = np.dot(np.dot(self.H, self.P), self.H.T) + self.R
        K = np.dot(np.dot(self.P, self.H.T), np.linalg.inv(S))
        self.x = self.x + np.dot(K, y)
        self.P = self.P - np.dot(np.dot(K, self.H), self.P)

class Track:
    def __init__(self, bbox, track_id):
        self.id = track_id
        self.hits = 0
        self.age = 0
        self.time_since_update = 0
        self.bbox = bbox
        self.kf = KalmanFilter(dim_x=7, dim_z=4)
        self.kf.F = np.array([[1,0,0,0,1,0,0],[0,1,0,0,0,1,0],[0,0,1,0,0,0,1],[0,0,0,1,0,0,0],[0,0,0,0,1,0,0],[0,0,0,0,0,1,0],[0,0,0,0,0,0,1]])
        self.kf.H = np.array([[1,0,0,0,0,0,0],[0,1,0,0,0,0,0],[0,0,1,0,0,0,0],[0,0,0,1,0,0,0]])
        (x, y, w, h) = bbox
        self.kf.x = np.array([[x+w/2],[y+h/2],[w*h],[w/h],[0],[0],[0]])
        self.kf.P *= 10.; self.kf.R *= 10.; self.kf.Q[4:, 4:] *= 0.01

    def update(self, bbox):
        self.time_since_update = 0
        self.hits += 1
        self.bbox = bbox
        (x, y, w, h) = bbox
        self.kf.update(np.array([[x+w/2],[y+h/2],[w*h],[w/h]]))

    def predict(self):
        self.kf.predict()
        self.age += 1
        self.time_since_update += 1
        return self.get_bbox()

    def get_bbox(self):
        x, y, s, r = self.kf.x[:4].flatten()
        w = np.sqrt(s * r); h = s / w
        return [x - w / 2, y - h / 2, w, h]

class Sort:
    def __init__(self, max_age=1, min_hits=3):
        self.max_age = max_age
        self.min_hits = min_hits
        self.tracks = []; self.frame_count = 0; self.track_id_counter = 0

    def update(self, detections):
        self.frame_count += 1
        predicted_tracks = [track.predict() for track in self.tracks]
        detections = np.array(detections) if len(detections) > 0 else np.empty((0, 4))
        predicted_tracks = np.array(predicted_tracks) if len(predicted_tracks) > 0 else np.empty((0, 4))
        iou_matrix = self._iou_batch(predicted_tracks, detections)
        row_ind, col_ind = linear_sum_assignment(-iou_matrix)
        for r, c in zip(row_ind, col_ind):
            if iou_matrix[r, c] >= 0.3: self.tracks[r].update(detections[c])
        unmatched_detections = [d for i, d in enumerate(detections) if i not in col_ind]
        for bbox in unmatched_detections:
            self.track_id_counter += 1
            self.tracks.append(Track(bbox, self.track_id_counter))
        self.tracks = [t for t in self.tracks if t.time_since_update <= self.max_age]
        output_tracks = []
        for track in self.tracks:
            if track.hits >= self.min_hits or self.frame_count <= self.min_hits:
                output_tracks.append(np.concatenate((track.get_bbox(), [track.id])).reshape(1, -1))
        return np.concatenate(output_tracks) if len(output_tracks) > 0 else np.empty((0, 5))

    def _iou_batch(self, bboxes1, bboxes2):
        if bboxes1.shape[0] == 0 or bboxes2.shape[0] == 0: return np.zeros((bboxes1.shape[0], bboxes2.shape[0]))
        bboxes1 = np.hstack((bboxes1[:, :2], bboxes1[:, :2] + bboxes1[:, 2:])) # convert x,y,w,h to x1,y1,x2,y2
        bboxes2 = np.hstack((bboxes2[:, :2], bboxes2[:, :2] + bboxes2[:, 2:]))
        x1 = np.maximum(bboxes1[:, None, 0], bboxes2[:, 0]); y1 = np.maximum(bboxes1[:, None, 1], bboxes2[:, 1])
        x2 = np.minimum(bboxes1[:, None, 2], bboxes2[:, 2]); y2 = np.minimum(bboxes1[:, None, 3], bboxes2[:, 3])
        intersection = np.maximum(0, x2 - x1) * np.maximum(0, y2 - y1)

        #union calculation
        area1 = (bboxes1[:, 2] - bboxes1[:, 0]) * (bboxes1[:, 3] - bboxes1[:, 1])
        area2 = (bboxes2[:, 2] - bboxes2[:, 0]) * (bboxes2[:, 3] - bboxes2[:, 1])
        union = area1[:, None] + area2[None, :] - intersection

        return intersection / union

def process_video(input_path, output_path):
    model = YOLO('yolov8n.pt')
    tracker = Sort()
    cap = cv2.VideoCapture(input_path)
    width, height, fps = int(cap.get(3)), int(cap.get(4)), int(cap.get(5))
    out = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break
        results = model(frame, verbose=False)
        detections = []
        for r in results:
            for *xyxy, conf, cls in r.boxes.data.tolist():
                x1, y1, x2, y2 = map(int, xyxy)
                # Only append the bbox coordinates, not the confidence score
                detections.append([x1, y1, x2-x1, y2-y1])

        tracks = tracker.update(np.array(detections))
        for x, y, w, h, obj_id in tracks:
            cv2.rectangle(frame, (int(x), int(y)), (int(x+w), int(y+h)), (0, 255, 0), 2)
            cv2.putText(frame, f"ID: {int(obj_id)}", (int(x), int(y-10)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
        out.write(frame)

    cap.release(); out.release()
    print(f"Done! Saved to {output_path}")
    files.download(output_path)

# Upload and run
uploaded = files.upload()
input_file = list(uploaded.keys())[0]
process_video(input_file, 'output_tracked.mp4')